# SISH adversarial miner — v2

Closed-loop search for **real Kannada words** whose plausible Romanizations may pull a target model toward an unrelated English concept.

### What this version fixes

- **WordNet setup is deterministic and verified before model loading.**
- Removed the duplicate silent `nltk.download(...)` call that caused the earlier `LookupError`.
- `cementu -> cement`-style loanword / semantic-overlap cases are filtered before evaluation.
- Semantic testing uses a **WordNet-derived semantic attractor**, not just the literal colliding substring.
- Candidate batches are diversified across native lexical roots and attractor tokens.
- Likelihood-margin positives are followed by neutral free-generation verification.
- Automatic positives are exported as **manual-review candidates**, not declared verified SISH examples automatically.

### Recommended execution

Run the notebook **top to bottom in a fresh Colab GPU runtime**.

Continue only after the WordNet cell prints:

`WORDNET PREFLIGHT: PASS`


In [1]:
!pip -q install -U "transformers>=4.55,<5" accelerate bitsandbytes sentencepiece wordfreq Unidecode pandas tqdm PyYAML huggingface_hub nltk


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Robust WordNet preflight.
# This cell must succeed BEFORE the model or miner is loaded.

import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

import nltk

NLTK_DIR = Path("/content/nltk_data")
CORPORA_DIR = NLTK_DIR / "corpora"
NLTK_DIR.mkdir(parents=True, exist_ok=True)
CORPORA_DIR.mkdir(parents=True, exist_ok=True)

os.environ["NLTK_DATA"] = str(NLTK_DIR)
if str(NLTK_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DIR))


def _wordnet_files_present():
    """Return True only when a usable WordNet corpus is physically present."""
    extracted = CORPORA_DIR / "wordnet"
    required = [
        extracted / "lexnames",
        extracted / "index.noun",
        extracted / "data.noun",
    ]
    if all(path.exists() for path in required):
        return True

    zipped = CORPORA_DIR / "wordnet.zip"
    if zipped.exists() and zipfile.is_zipfile(zipped):
        try:
            with zipfile.ZipFile(zipped) as zf:
                names = set(zf.namelist())
                return (
                    "wordnet/lexnames" in names
                    and "wordnet/index.noun" in names
                    and "wordnet/data.noun" in names
                )
        except zipfile.BadZipFile:
            return False

    return False


def _download_wordnet_with_nltk():
    print("Trying the NLTK WordNet downloader...")
    try:
        ok = nltk.download(
            "wordnet",
            download_dir=str(NLTK_DIR),
            quiet=False,
            raise_on_error=False,
        )
        return bool(ok) and _wordnet_files_present()
    except Exception as exc:
        print(f"NLTK downloader failed: {type(exc).__name__}: {exc}")
        return False


def _download_wordnet_direct():
    """Fallback to the official NLTK data archive."""
    url = (
        "https://raw.githubusercontent.com/nltk/nltk_data/"
        "gh-pages/packages/corpora/wordnet.zip"
    )
    destination = CORPORA_DIR / "wordnet.zip"
    temporary = CORPORA_DIR / "wordnet.zip.part"

    print("Trying direct download from the official NLTK data repository...")

    if temporary.exists():
        temporary.unlink()

    request = urllib.request.Request(
        url,
        headers={"User-Agent": "SISH-adversarial-miner/2.0"},
    )

    try:
        with urllib.request.urlopen(request, timeout=180) as response, temporary.open("wb") as out:
            shutil.copyfileobj(response, out)
    except Exception as exc:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            "WordNet could not be downloaded. Check that this Colab runtime "
            "has internet access, then rerun this cell."
        ) from exc

    if not zipfile.is_zipfile(temporary):
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            "The WordNet download was not a valid ZIP file. "
            "The server may have returned an HTML error page."
        )

    temporary.replace(destination)

    # Extract as well as keep the ZIP. This avoids loader differences across
    # NLTK/Python versions.
    extracted = CORPORA_DIR / "wordnet"
    if extracted.exists():
        shutil.rmtree(extracted)

    with zipfile.ZipFile(destination) as zf:
        zf.extractall(CORPORA_DIR)

    if not _wordnet_files_present():
        raise RuntimeError(
            "WordNet was downloaded but the expected corpus files were not found."
        )


if not _wordnet_files_present():
    if not _download_wordnet_with_nltk():
        _download_wordnet_direct()

# Import WordNet only after its data exists. This avoids leaving the
# LazyCorpusLoader in a failed state.
from nltk.corpus import wordnet as wn

try:
    wn.ensure_loaded()
    _probe_synsets = wn.synsets("inn")
except LookupError as exc:
    raise RuntimeError(
        "WordNet files exist, but NLTK still cannot load them. "
        f"NLTK search paths: {nltk.data.path}"
    ) from exc

if not _probe_synsets:
    raise RuntimeError(
        "WordNet loaded, but the sanity-check word 'inn' returned no synsets."
    )

print("WORDNET PREFLIGHT: PASS")
print("NLTK data directory:", NLTK_DIR)
print("WordNet sanity check:", _probe_synsets[:3])


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Trying the NLTK WordNet downloader...


[nltk_data] Downloading package wordnet to /content/nltk_data...
[nltk_data]   Unzipping corpora/wordnet.zip.


WORDNET PREFLIGHT: PASS
NLTK data directory: /content/nltk_data
WordNet sanity check: [Synset('hostel.n.01')]


In [3]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
SEED = 17

# A 12-word screen is too noisy (one item changes accuracy by 8.3%).
# 200 gives a much more useful first-pass estimate while remaining practical.
SCREEN_WORDS = 200
SCREEN_CHOICES = 4
MIN_NATIVE_SCREEN_ACCURACY = 0.50
MIN_SCRIPT_GAP = 0.10

GENERATIONS = 3
PREFILTER_PER_GENERATION = 32
PARENTS_PER_GENERATION = 12
VARIANTS_PER_PARENT = 24
MAX_PER_ATTRACTOR_PER_GENERATION = 2
MAX_TESTS_PER_NATIVE = 2
NOVELTY_BONUS = 0.35

PROMPT_WINS_REQUIRED = 2
GENERATION_WINS_REQUIRED = 1
GENERATION_PROMPTS_TO_TEST = 3
GENERATION_MAX_NEW_TOKENS = 16

MAX_GLOSSES_PER_WORD = 4
REQUIRE_INDOWORDNET = False
REQUIRE_INDICCORP_ATTESTATION = False
ROMANIZATION_SOURCES = {"Dakshina"}

MIN_ENGLISH_ZIPF = 2.8
MAX_SEMANTIC_OVERLAP = 0.20
MIN_SHARED_CONCEPT_STEMS_TO_REJECT = 2

ALAR_REVISION = "8651ccf8e92184ca17e234eeb6c947d8d52dd5c4"
AKSHARANTAR_REVISION = "e418c1fc928d9f5393af33268472cf20c1891be8"
INDOWORDNET_REVISION = "6dfbbdc8bf9da9500f8eaa2eeb13f150186941d0"

# IMPORTANT: the primary prompts are deliberately neutral. Saying "Kannada"
# in the prompt can itself repair language ambiguity and suppress the effect.
PROMPT_TEMPLATES = [
    "Give the English meaning of this word. Reply with one short gloss.\\nWord: {word}\\nMeaning:",
    "Translate the following word into English using a brief dictionary gloss.\\n{word}\\nEnglish:",
    "What does this word mean in English? Give only a brief gloss.\\n{word}\\nAnswer:",
]

# Optional language-label control.
LABELED_CONTROL_TEMPLATE = (
    "The following is a Kannada word written in its current script. "
    "Give its English meaning using one short gloss.\\nWord: {word}\\nMeaning:"
)


In [4]:
import json
import math
import random
import re
import zipfile
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from huggingface_hub import hf_hub_download
from nltk.corpus import wordnet as wn
from nltk.stem import PorterStemmer
from tqdm.auto import tqdm
from unidecode import unidecode
from urllib.request import Request, urlopen
from wordfreq import zipf_frequency
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# The WordNet preflight cell above must have succeeded.
try:
    wn.ensure_loaded()
except LookupError as exc:
    raise RuntimeError(
        "WordNet preflight was not completed. Run the WordNet preflight cell "
        "above before loading the model."
    ) from exc


random.seed(SEED)
np.random.seed(SEED)
assert torch.cuda.is_available(), "Switch Colab to a GPU runtime, then run again."

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.eval()
print(f"Loaded {MODEL_ID} on {model.get_input_embeddings().weight.device}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

... (5 outputs left out)

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded mistralai/Mistral-7B-Instruct-v0.3 on cuda:0


## Lexicon

Meanings come from [Alar](https://github.com/alar-dict/data), an open Kannada–English dictionary created by lexicographer V. Krishna. Romanizations are restricted to the Dakshina portion of [Aksharantar](https://huggingface.co/datasets/ai4bharat/Aksharantar). IndoWordNet and IndicCorp membership are kept as independent provenance checks. No model generates Kannada words or gold meanings.


In [5]:
source_dir = Path("/content/sish_lexicon_sources")
source_dir.mkdir(exist_ok=True)


def download(url, destination):
    if destination.exists():
        return destination
    request = Request(url, headers={"User-Agent": "SISH-research-notebook"})
    with urlopen(request, timeout=180) as response, destination.open("wb") as output:
        while chunk := response.read(1024 * 1024):
            output.write(chunk)
    return destination


alar_path = download(
    f"https://raw.githubusercontent.com/alar-dict/data/{ALAR_REVISION}/alar.yml",
    source_dir / "alar.yml",
)
aksharantar_path = Path(hf_hub_download(
    repo_id="ai4bharat/Aksharantar",
    filename="kan.zip",
    repo_type="dataset",
    revision=AKSHARANTAR_REVISION,
))
indowordnet_path = Path(hf_hub_download(
    repo_id="cfilt/iwn_wordlists",
    filename="kannada.json",
    repo_type="dataset",
    revision=INDOWORDNET_REVISION,
))


def compact_gloss(text, max_words=14):
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = re.sub(r"^(?:\([^)]*\)\s*)+", "", text)
    text = re.sub(r"^\d+[.)]\s*", "", text)
    text = re.split(r"[;.]", text, maxsplit=1)[0].strip(" ,:-")
    if not re.search(r"[A-Za-z]", text) or re.search(r"[\u0C80-\u0CFF]", text):
        return None
    words = text.split()
    return " ".join(words[:max_words]) if words else None


with alar_path.open(encoding="utf-8") as handle:
    alar_data = yaml.safe_load(handle)

alar = {}
for item in alar_data:
    native = str(item.get("entry", "")).strip()
    if not native or not all("\u0C80" <= char <= "\u0CFF" for char in native):
        continue
    senses = []
    parts_of_speech = []
    for definition in item.get("defs", []):
        gloss = compact_gloss(definition.get("entry", ""))
        if gloss and gloss not in senses:
            senses.append(gloss)
        pos = str(definition.get("type", "")).strip()
        if pos and pos not in parts_of_speech:
            parts_of_speech.append(pos)
    if not senses:
        continue
    record = alar.setdefault(native, {
        "dictionary_ids": [], "gold_glosses": [], "parts_of_speech": []
    })
    record["dictionary_ids"].append(str(item.get("id", "")))
    record["gold_glosses"].extend(g for g in senses if g not in record["gold_glosses"])
    record["parts_of_speech"].extend(p for p in parts_of_speech if p not in record["parts_of_speech"])
del alar_data

iwn_payload = json.loads(indowordnet_path.read_text(encoding="utf-8"))
iwn_words = {row["word"] for row in iwn_payload["data"]}

roman_counts = defaultdict(int)
indiccorp_words = set()
with zipfile.ZipFile(aksharantar_path) as archive:
    for member in archive.namelist():
        if not member.endswith(".json"):
            continue
        with archive.open(member) as handle:
            for line in handle:
                row = json.loads(line)
                native = row["native word"].strip()
                source = row["source"]
                if source == "IndicCorp" and native in alar:
                    indiccorp_words.add(native)
                if native not in alar or source not in ROMANIZATION_SOURCES:
                    continue
                roman = re.sub(r"[^a-z]", "", unidecode(row["english word"]).lower())
                if 2 <= len(roman) <= 40:
                    roman_counts[(native, roman, source)] += 1

rows = []
for (native, roman, source), attested_count in roman_counts.items():
    entry = alar[native]
    glosses = entry["gold_glosses"]
    if not glosses or len(glosses) > MAX_GLOSSES_PER_WORD:
        continue
    rows.append({
        "native": native,
        "roman": roman,
        "gloss": glosses[0],
        "gold_glosses": json.dumps(glosses, ensure_ascii=False),
        "all_glosses": " | ".join(entry["gold_glosses"]),
        "gloss_count": len(entry["gold_glosses"]),
        "part_of_speech": " | ".join(entry["parts_of_speech"]),
        "dictionary_ids": " | ".join(entry["dictionary_ids"]),
        "lexicon_source": "Alar",
        "romanization_source": source,
        "roman_attested_count": attested_count,
        "in_indowordnet": native in iwn_words,
        "in_indiccorp": native in indiccorp_words,
    })

lexicon = pd.DataFrame(rows)
if REQUIRE_INDOWORDNET:
    lexicon = lexicon[lexicon["in_indowordnet"]]
if REQUIRE_INDICCORP_ATTESTATION:
    lexicon = lexicon[lexicon["in_indiccorp"]]

lexicon = lexicon.drop_duplicates(["native", "roman"]).reset_index(drop=True)
assert len(lexicon) >= 100, "The source intersection is unexpectedly small; inspect the pinned datasets."
assert set(lexicon["lexicon_source"]) == {"Alar"}

print(f"{len(lexicon):,} dictionary words with attested Romanizations")
print(f"IndoWordNet overlap: {lexicon.in_indowordnet.mean():.1%}")
print(f"IndicCorp overlap: {lexicon.in_indiccorp.mean():.1%}")
display(lexicon.sample(min(8, len(lexicon)), random_state=SEED))


kan.zip:   0%|          | 0.00/87.6M [00:00<?, ?B/s]

kannada.json: 0.00B [00:00, ?B/s]

5,735 dictionary words with attested Romanizations
IndoWordNet overlap: 64.3%
IndicCorp overlap: 8.3%


,native,roman,gloss,gold_glosses,all_glosses,gloss_count,part_of_speech,dictionary_ids,lexicon_source,romanization_source,roman_attested_count,in_indowordnet,in_indiccorp
2537,ಬಹುವಾರ್ಷಿಕ,bahuvarshika,having a life cycle lasting more than two years,"[""having a life cycle lasting more than two ye...",having a life cycle lasting more than two year...,2,adjective | noun,108105 | 108106,Alar,Dakshina,1,True,False
3325,ಸಾವಿರ,saavira,amounting to one thousand in number,"[""amounting to one thousand in number"", ""too n...",amounting to one thousand in number | too nume...,3,adjective | noun,144959 | 144960,Alar,Dakshina,1,True,False
4352,ಸ್ವೀಕಾರ,swiikaara,the act of receiving,"[""the act of receiving"", ""the act of taking in...",the act of receiving | the act of taking in in...,3,noun,150190,Alar,Dakshina,1,True,False
4782,ಆದಾಗ್ಯೂ,aadaagyoo,in spite of,"[""in spite of""]",in spite of,1,preposition,16420,Alar,Dakshina,1,False,False
350,ಶತಪಥ,shatapatha,"the well known brāhmaṇa (Samskřta text, contai...","[""the well known brāhmaṇa (Samskřta text, cont...","the well known brāhmaṇa (Samskřta text, contai...",2,noun,136377 | 136378,Alar,Dakshina,1,False,False
4265,ಖರೀದಿ,kharidi,the act of obtaining (something) for money or ...,"[""the act of obtaining (something) for money o...",the act of obtaining (something) for money or ...,1,noun,53261,Alar,Dakshina,1,False,True
1682,ಮಮತೆ,mamate,love or affection one has towards the younger ...,"[""love or affection one has towards the younge...",love or affection one has towards the younger ...,1,noun,117296,Alar,Dakshina,1,True,False
3462,ಸೇಡು,sedu,the act of shrinking one's size,"[""the act of shrinking one's size"", ""the quali...",the act of shrinking one's size | the quality ...,4,noun,148245 | 148246,Alar,Dakshina,1,True,False


In [6]:
def render_prompt(word, template_id=0):
    user_text = PROMPT_TEMPLATES[template_id].format(word=word)
    messages = [{"role": "user", "content": user_text}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
    )


def render_labeled_control(word):
    user_text = LABELED_CONTROL_TEMPLATE.format(word=word)
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=True,
        add_generation_prompt=True,
    )


@torch.inference_mode()
def completion_logprobs(prompts, answers, batch_size=12):
    """Length-normalized mean token log-probability of each answer."""
    device = model.get_input_embeddings().weight.device
    results = []

    for start in tqdm(range(0, len(prompts), batch_size), leave=False):
        batch_prompts = prompts[start:start + batch_size]
        batch_answers = answers[start:start + batch_size]
        rows, prompt_lengths, answer_lengths = [], [], []

        for prompt_ids, answer in zip(batch_prompts, batch_answers):
            prompt_ids = list(prompt_ids)
            answer_ids = tokenizer.encode(" " + answer.strip(), add_special_tokens=False)
            if not answer_ids:
                answer_ids = [tokenizer.unk_token_id]
            rows.append(prompt_ids + answer_ids)
            prompt_lengths.append(len(prompt_ids))
            answer_lengths.append(len(answer_ids))

        width = max(map(len, rows))
        input_ids = torch.full((len(rows), width), tokenizer.pad_token_id, dtype=torch.long)
        attention = torch.zeros_like(input_ids)
        for i, row in enumerate(rows):
            input_ids[i, :len(row)] = torch.tensor(row)
            attention[i, :len(row)] = 1

        input_ids = input_ids.to(device)
        attention = attention.to(device)
        logits = model(input_ids=input_ids, attention_mask=attention).logits.float()
        log_probs = logits.log_softmax(dim=-1)

        for i, (prompt_len, answer_len) in enumerate(zip(prompt_lengths, answer_lengths)):
            target = input_ids[i, prompt_len:prompt_len + answer_len]
            prediction = log_probs[i, prompt_len - 1:prompt_len + answer_len - 1]
            token_scores = prediction.gather(1, target.unsqueeze(1)).squeeze(1)
            results.append(token_scores.mean().item())

    return results


@torch.inference_mode()
def generate_short_answer(word, template_id=0):
    """Greedy free generation used only to verify margin-positive candidates."""
    device = model.get_input_embeddings().weight.device
    prompt_ids = render_prompt(word, template_id)
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    attention_mask = torch.ones_like(input_ids)

    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        do_sample=False,
        max_new_tokens=GENERATION_MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    continuation = output[0, input_ids.shape[1]:]
    return tokenizer.decode(continuation, skip_special_tokens=True).strip()


In [7]:
def screen_target(frame, n=SCREEN_WORDS, choices=SCREEN_CHOICES):
    # Prefer more strongly attested Romanizations for the capability screen.
    ordered = frame.sort_values(
        ["in_indiccorp", "roman_attested_count"],
        ascending=[False, False],
    )
    subset = ordered.drop_duplicates("native").head(min(n, len(ordered))).reset_index(drop=True)

    glosses = frame["gloss"].drop_duplicates().tolist()
    if len(glosses) < choices:
        raise ValueError(f"Screening needs at least {choices} distinct glosses.")

    rng = random.Random(SEED)
    jobs = []
    for row_id, row in subset.iterrows():
        distractors = rng.sample([g for g in glosses if g != row.gloss], choices - 1)
        options = [row.gloss] + distractors
        rng.shuffle(options)
        for script, word in [("native", row.native), ("roman", row.roman)]:
            for option in options:
                jobs.append((row_id, script, word, row.gloss, option))

    scores = completion_logprobs(
        [render_prompt(job[2]) for job in jobs],
        [job[4] for job in jobs],
    )
    scored = pd.DataFrame(jobs, columns=["row_id", "script", "word", "gold", "option"])
    scored["score"] = scores
    picks = scored.loc[scored.groupby(["row_id", "script"])["score"].idxmax()].copy()
    picks["correct"] = picks["gold"] == picks["option"]

    accuracy = picks.groupby("script")["correct"].mean()
    native_acc = float(accuracy.get("native", 0.0))
    roman_acc = float(accuracy.get("roman", 0.0))
    summary = pd.DataFrame([{
        "model": MODEL_ID,
        "words": len(subset),
        "native_accuracy": native_acc,
        "roman_accuracy": roman_acc,
        "script_gap": native_acc - roman_acc,
        "chance_accuracy": 1.0 / choices,
    }])
    return summary, picks


screen_summary, screen_details = screen_target(lexicon)
display(screen_summary.round(3))

native_acc = float(screen_summary.loc[0, "native_accuracy"])
script_gap = float(screen_summary.loc[0, "script_gap"])
if native_acc < MIN_NATIVE_SCREEN_ACCURACY:
    print(
        "WARNING: native-script knowledge is weak on this screen. "
        "Individual counterfactuals may still be interesting, but this model is a weak primary mining target."
    )
elif script_gap < MIN_SCRIPT_GAP:
    print(
        "WARNING: the aggregate script gap is small. "
        "Continue only if individual Romanized/native counterfactuals are strong."
    )
else:
    print("Screen passed the configured native-knowledge and script-gap checks.")


  0%|          | 0/134 [00:00<?, ?it/s]

,model,words,native_accuracy,roman_accuracy,script_gap,chance_accuracy
0,mistralai/Mistral-7B-Instruct-v0.3,200,0.385,0.38,0.005,0.25


In [8]:
REPLACEMENTS = [
    ("aa", "a", "vowel_length"),
    ("ee", "e", "vowel_length"),
    ("ii", "i", "vowel_length"),
    ("oo", "o", "vowel_length"),
    ("uu", "u", "vowel_length"),
    ("sh", "s", "sibilant"),
    ("v", "w", "v_w"),
    ("ph", "f", "aspirate"),
    ("bh", "b", "aspirate"),
    ("th", "t", "aspirate"),
    ("dh", "d", "aspirate"),
    ("kh", "k", "aspirate"),
    ("gh", "g", "aspirate"),
    ("ch", "c", "affricate"),
]


def mutate_once(text):
    variants = {}
    for left, right, rule in REPLACEMENTS:
        for source, target in [(left, right), (right, left)]:
            for match in re.finditer(re.escape(source), text):
                i = match.start()
                candidate = text[:i] + target + text[i + len(source):]
                if candidate != text:
                    variants[candidate] = rule

    collapsed = re.sub(r"([bcdfghjklmnpqrstvwxyz])\1", r"\1", text)
    if collapsed != text:
        variants[collapsed] = "consonant_length"

    for match in re.finditer(r"[kptmnlr]", text):
        i = match.start()
        if (i == 0 or text[i - 1] != text[i]) and (i + 1 == len(text) or text[i + 1] != text[i]):
            variants[text[:i] + text[i] + text[i:]] = "consonant_length"

    return list(variants.items())


def clean_piece(piece):
    # SentencePiece/BPE tokenizers mark word starts with ▁, Ġ, or (in some displays) "_".
    piece = (
        str(piece)
        .replace("▁", "")
        .replace("Ġ", "")
        .lstrip("_")
        .lower()
        .strip()
    )
    return piece if re.fullmatch(r"[a-z]+", piece or "") else ""


# Build an empirical Romanized-Kannada subpiece frequency table from the
# attested Aksharantar lexicon. Comparing Latin pieces with wordfreq(..., "kn")
# is not meaningful because "kn" is a native-script corpus.
roman_piece_counts = Counter()
for row in lexicon.itertuples():
    weight = max(1, int(row.roman_attested_count))
    for raw_piece in tokenizer.tokenize(row.roman):
        piece = clean_piece(raw_piece)
        if piece:
            roman_piece_counts[piece] += weight

NON_LEXICAL_PIECES = {
    "the", "and", "for", "with", "ing", "tion", "ment", "ness",
    "able", "ible", "ive", "ally", "est", "ous", "ize", "ise",
}

STOPWORDS = {
    "a", "an", "the", "of", "to", "in", "on", "for", "with", "and", "or",
    "that", "which", "is", "are", "be", "being", "made", "having", "one",
    "something", "someone", "thing", "person", "quality", "state", "act",
}
stemmer = PorterStemmer()


def content_stems(text):
    words = re.findall(r"[a-z]+", str(text).lower())
    return {
        stemmer.stem(word)
        for word in words
        if len(word) >= 3 and word not in STOPWORDS
    }


@lru_cache(maxsize=10000)
def english_concept(word):
    """Return a non-copy semantic paraphrase and a WordNet concept family.

    WordNet is preflighted at notebook startup. This guard keeps any later
    environment/path failure from surfacing as an opaque pandas.map traceback.
    """
    try:
        synsets = wn.synsets(word)
    except LookupError as exc:
        raise RuntimeError(
            "WordNet became unavailable during candidate mining. "
            "Rerun the WordNet preflight cell and then rerun the cells below it."
        ) from exc

    if not synsets:
        return None

    syn = synsets[0]
    family_phrases = set()
    for candidate_syn in synsets[:3]:
        for lemma in candidate_syn.lemmas():
            family_phrases.add(lemma.name().replace("_", " ").lower())
        family_phrases.add(candidate_syn.definition().lower())

    # Prefer an alternative lemma from the same sense.
    alternatives = []
    for lemma in syn.lemmas():
        phrase = lemma.name().replace("_", " ").lower()
        if phrase != word.lower() and word.lower() not in phrase.split():
            alternatives.append(phrase)

    # If there is no synonym, use a short hypernym.
    if not alternatives:
        for hypernym in syn.hypernyms():
            for lemma in hypernym.lemmas():
                phrase = lemma.name().replace("_", " ").lower()
                if phrase != word.lower() and word.lower() not in phrase.split():
                    alternatives.append(phrase)
                    family_phrases.add(phrase)

    if alternatives:
        alternatives = sorted(
            set(alternatives),
            key=lambda x: (-zipf_frequency(x.split()[0], "en"), len(x)),
        )
        paraphrase = alternatives[0]
        source = "wordnet_lemma_or_hypernym"
    else:
        definition_words = [
            w for w in re.findall(r"[a-z]+", syn.definition().lower())
            if w not in STOPWORDS and w != word.lower()
        ]
        paraphrase = " ".join(definition_words[:6]).strip()
        source = "wordnet_definition"

    if not paraphrase:
        return None

    family_phrases.add(word.lower())
    family_phrases.add(paraphrase)
    family_tokens = sorted({
        token
        for phrase in family_phrases
        for token in re.findall(r"[a-z]+", phrase)
        if len(token) >= 3 and token not in STOPWORDS
    })

    concept_text = " ".join(sorted(family_phrases))
    return {
        "paraphrase": paraphrase,
        "family_tokens": family_tokens,
        "concept_text": concept_text,
        "source": source,
        "definition": syn.definition(),
    }


def semantic_overlap(all_glosses, concept):
    gold = content_stems(all_glosses)
    concept_stem_set = content_stems(concept["concept_text"])
    if not gold or not concept_stem_set:
        return 0.0, 0
    shared = gold & concept_stem_set
    overlap = len(shared) / max(1, min(len(gold), len(concept_stem_set)))
    return float(overlap), len(shared)


def collision_features(roman):
    pieces = tokenizer.tokenize(roman)
    candidates = []

    for raw_piece in pieces:
        piece = clean_piece(raw_piece)
        if len(piece) < 3 or piece in NON_LEXICAL_PIECES:
            continue

        # Requiring a WordNet sense removes many arbitrary BPE fragments.
        concept = english_concept(piece)
        if concept is None:
            continue

        en_zipf = zipf_frequency(piece, "en")
        if en_zipf < MIN_ENGLISH_ZIPF:
            continue

        kn_piece_count = roman_piece_counts.get(piece, 0)

        # High when common in English and rare as an attested Kannada Romanization subpiece.
        # This is a ranking score, not a calibrated probability.
        kn_penalty = math.log10(1.0 + kn_piece_count)
        score = en_zipf - kn_penalty
        candidates.append((score, en_zipf, kn_piece_count, piece, concept))

    if not candidates:
        return (
            " | ".join(pieces), None, None, None, -99.0, 0.0, 0,
            "[]", "", ""
        )

    score, en_zipf, kn_piece_count, collision_token, concept = max(candidates, key=lambda x: x[0])
    return (
        " | ".join(pieces),
        collision_token,
        concept["paraphrase"],
        concept["source"],
        float(score),
        float(en_zipf),
        int(kn_piece_count),
        json.dumps(concept["family_tokens"]),
        concept["definition"],
        concept["concept_text"],
    )


In [9]:
# Semantic-attractor preflight.
# This exercises the same WordNet access path used later by collision_features().

for probe_word in ["inn", "cement", "math", "bad"]:
    concept = english_concept(probe_word)
    print(
        probe_word,
        "->",
        None if concept is None else {
            "paraphrase": concept["paraphrase"],
            "source": concept["source"],
            "definition": concept["definition"],
        },
    )

assert english_concept("inn") is not None, "Semantic-attractor preflight failed."
print("SEMANTIC ATTRACTOR PREFLIGHT: PASS")


inn -> {'paraphrase': 'lodge', 'source': 'wordnet_lemma_or_hypernym', 'definition': 'a hotel providing overnight lodging for travelers'}
cement -> {'paraphrase': 'concrete', 'source': 'wordnet_lemma_or_hypernym', 'definition': 'concrete pavement is sometimes referred to as cement'}
math -> {'paraphrase': 'mathematics', 'source': 'wordnet_lemma_or_hypernym', 'definition': 'a science (or group of related sciences) dealing with the logic of quantity and shape and arrangement'}
bad -> {'paraphrase': 'badness', 'source': 'wordnet_lemma_or_hypernym', 'definition': 'that which is below standard or expectations as of ethics or decency'}
SEMANTIC ATTRACTOR PREFLIGHT: PASS


In [10]:
def text_contains_concept(text, family_tokens):
    stems = content_stems(text)
    family_stems = {stemmer.stem(token) for token in family_tokens}
    return bool(stems & family_stems)


def verify_free_generations(frame):
    """Run free generation only on margin-positive candidates.

    This is intentionally a conservative automatic screen. Saved outputs must
    still be manually inspected before a paper labels an example as verified SISH.
    """
    frame = frame.copy()

    # Pre-create columns with object dtype so JSON strings assign cleanly.
    frame["roman_generation_outputs"] = "[]"
    frame["native_generation_outputs"] = "[]"
    frame["roman_generation_attractor_hits"] = 0
    frame["native_generation_attractor_hits"] = 0
    frame["auto_strict_positive"] = False

    for idx, row in frame.iterrows():
        if not bool(row.margin_positive):
            continue

        family_tokens = json.loads(row.attractor_family)
        roman_outputs, native_outputs = [], []
        roman_hits = native_hits = 0

        for template_id in range(min(GENERATION_PROMPTS_TO_TEST, len(PROMPT_TEMPLATES))):
            roman_answer = generate_short_answer(row.roman, template_id)
            native_answer = generate_short_answer(row.native, template_id)
            roman_outputs.append(roman_answer)
            native_outputs.append(native_answer)
            roman_hits += int(text_contains_concept(roman_answer, family_tokens))
            native_hits += int(text_contains_concept(native_answer, family_tokens))

        frame.at[idx, "roman_generation_outputs"] = json.dumps(roman_outputs, ensure_ascii=False)
        frame.at[idx, "native_generation_outputs"] = json.dumps(native_outputs, ensure_ascii=False)
        frame.at[idx, "roman_generation_attractor_hits"] = roman_hits
        frame.at[idx, "native_generation_attractor_hits"] = native_hits
        frame.at[idx, "auto_strict_positive"] = bool(
            row.margin_positive
            and roman_hits >= GENERATION_WINS_REQUIRED
            and native_hits == 0
        )

    frame["auto_strict_positive"] = frame["auto_strict_positive"].astype(bool)
    return frame


def evaluate_candidates(frame):
    frame = frame.reset_index(drop=True).copy()
    jobs = []

    for candidate_id, row in frame.iterrows():
        gold_glosses = json.loads(row.gold_glosses)

        for script, word in [("roman", row.roman), ("native", row.native)]:
            for template_id in range(len(PROMPT_TEMPLATES)):
                prompt = render_prompt(word, template_id)
                for gloss_id, gloss in enumerate(gold_glosses):
                    jobs.append((candidate_id, script, template_id, f"true_{gloss_id}", prompt, gloss))

                # Compare against a semantic paraphrase such as "hotel/lodging",
                # not the literal collision token "inn".
                jobs.append((
                    candidate_id, script, template_id, "attractor",
                    prompt, row.predicted_attractor
                ))

    scores = completion_logprobs([job[4] for job in jobs], [job[5] for job in jobs])
    lookup = {(j[0], j[1], j[2], j[3]): score for j, score in zip(jobs, scores)}

    records = []
    for candidate_id, row in frame.iterrows():
        gold_glosses = json.loads(row.gold_glosses)
        roman_margins, native_margins = [], []

        for template_id in range(len(PROMPT_TEMPLATES)):
            roman_gold = max(
                lookup[(candidate_id, "roman", template_id, f"true_{gloss_id}")]
                for gloss_id in range(len(gold_glosses))
            )
            native_gold = max(
                lookup[(candidate_id, "native", template_id, f"true_{gloss_id}")]
                for gloss_id in range(len(gold_glosses))
            )

            roman_margins.append(
                lookup[(candidate_id, "roman", template_id, "attractor")] - roman_gold
            )
            native_margins.append(
                lookup[(candidate_id, "native", template_id, "attractor")] - native_gold
            )

        roman_wins = sum(m > 0 for m in roman_margins)
        native_wins = sum(m < 0 for m in native_margins)
        roman_margin = float(np.mean(roman_margins))
        native_margin = float(np.mean(native_margins))

        margin_positive = bool(
            roman_wins >= PROMPT_WINS_REQUIRED
            and native_wins >= PROMPT_WINS_REQUIRED
            and roman_margin > 0
            and native_margin < 0
        )

        reward = (
            (roman_margin - native_margin)
            + 0.10 * float(row.collision_score)
            + NOVELTY_BONUS * float(row.is_novel_attractor)
            - float(row.unnaturalness)
        )

        record = row.to_dict()
        record.update({
            "roman_margin": roman_margin,
            "native_margin": native_margin,
            "script_contrast": roman_margin - native_margin,
            "roman_prompt_wins": roman_wins,
            "native_prompt_wins": native_wins,
            "margin_positive": margin_positive,
            "reward": reward,
            "roman_margins": json.dumps([round(x, 5) for x in roman_margins]),
            "native_margins": json.dumps([round(x, 5) for x in native_margins]),
        })
        records.append(record)

    scored = pd.DataFrame(records)
    scored = verify_free_generations(scored)
    return scored


class SISHMiner:
    def __init__(self, frame):
        self.seeds = frame.copy()
        self.history = pd.DataFrame()
        self.seen = set()
        self.rule_prior = defaultdict(float)
        self.native_test_counts = Counter()
        self.seen_attractors = set()

    def seed_parents(self):
        parents = self.seeds.rename(columns={"roman": "base_roman"}).copy()
        parents["roman"] = parents["base_roman"]
        parents["rules"] = "base"
        parents["edit_count"] = 0
        return parents

    def propose(self, parents, generation):
        proposals = []

        for _, parent in parents.iterrows():
            if self.native_test_counts[parent.native] >= MAX_TESTS_PER_NATIVE:
                continue

            base = parent.to_dict()
            base["generation"] = generation
            proposals.append(base)

            children = mutate_once(parent.roman)
            children.sort(key=lambda item: self.rule_prior[item[1]], reverse=True)

            for roman, rule in children[:VARIANTS_PER_PARENT]:
                child = parent.to_dict()
                child["roman"] = roman
                child["rules"] = parent.rules + "|" + rule
                child["edit_count"] = int(parent.edit_count) + 1
                child["generation"] = generation
                proposals.append(child)

        if not proposals:
            return pd.DataFrame()

        proposed = pd.DataFrame(proposals).drop_duplicates(["native", "roman"])
        proposed = proposed[
            ~proposed.apply(lambda row: (row.native, row.roman) in self.seen, axis=1)
        ].copy()
        if proposed.empty:
            return proposed

        feature_columns = [
            "token_pieces",
            "collision_token",
            "predicted_attractor",
            "attractor_source",
            "collision_score",
            "english_zipf",
            "kannada_piece_count",
            "attractor_family",
            "attractor_definition",
            "attractor_concept_text",
        ]
        features = proposed["roman"].map(collision_features)
        proposed[feature_columns] = pd.DataFrame(features.tolist(), index=proposed.index)

        proposed = proposed[proposed["collision_token"].notna()].copy()
        if proposed.empty:
            return proposed

        overlaps = proposed.apply(
            lambda row: semantic_overlap(
                row.all_glosses,
                {"concept_text": row.attractor_concept_text},
            ),
            axis=1,
        )
        proposed["semantic_overlap"] = [x[0] for x in overlaps]
        proposed["shared_concept_stems"] = [x[1] for x in overlaps]

        # Reject likely cognates/loanwords/semantically aligned collisions such
        # as cementu -> cement before using them as SISH candidates.
        proposed = proposed[
            (proposed["semantic_overlap"] <= MAX_SEMANTIC_OVERLAP)
            & (proposed["shared_concept_stems"] < MIN_SHARED_CONCEPT_STEMS_TO_REJECT)
        ].copy()
        if proposed.empty:
            return proposed

        proposed["unnaturalness"] = (
            0.35 * proposed["edit_count"]
            + (
                (proposed["roman"].str.len() - proposed["base_roman"].str.len()).abs()
                / proposed["base_roman"].str.len().clip(lower=1)
            )
        )
        proposed["last_rule"] = proposed["rules"].str.split("|").str[-1]
        proposed["is_novel_attractor"] = ~proposed["collision_token"].isin(self.seen_attractors)
        proposed["search_score"] = (
            proposed["collision_score"]
            - proposed["unnaturalness"]
            + NOVELTY_BONUS * proposed["is_novel_attractor"].astype(float)
            + proposed["last_rule"].map(self.rule_prior).fillna(0) * 0.25
        )
        return proposed.sort_values("search_score", ascending=False)

    def select_diverse_batch(self, proposed):
        if proposed.empty:
            return proposed

        selected = []
        native_in_batch = set()
        attractor_counts = Counter()

        for _, row in proposed.sort_values("search_score", ascending=False).iterrows():
            if len(selected) >= PREFILTER_PER_GENERATION:
                break
            if row.native in native_in_batch:
                continue
            if self.native_test_counts[row.native] >= MAX_TESTS_PER_NATIVE:
                continue
            if attractor_counts[row.collision_token] >= MAX_PER_ATTRACTOR_PER_GENERATION:
                continue

            selected.append(row.to_dict())
            native_in_batch.add(row.native)
            attractor_counts[row.collision_token] += 1

        return pd.DataFrame(selected)

    def learn(self, tested):
        if tested.empty:
            return

        learned = tested.groupby("last_rule")["reward"].mean()
        center = float(learned.mean())
        for rule, value in learned.items():
            self.rule_prior[rule] = float(value - center)

        for native in tested.native:
            self.native_test_counts[native] += 1
        self.seen_attractors.update(tested.collision_token.dropna().tolist())

    def choose_parents(self):
        if self.history.empty:
            return self.seed_parents()

        eligible = self.history[
            self.history["native"].map(self.native_test_counts) < MAX_TESTS_PER_NATIVE
        ].copy()

        if eligible.empty:
            unseen_natives = set(self.seeds.native) - set(self.native_test_counts)
            seeded = self.seed_parents()
            return seeded[seeded.native.isin(unseen_natives)].head(PARENTS_PER_GENERATION)

        return (
            eligible.sort_values(["auto_strict_positive", "reward"], ascending=[False, False])
            .drop_duplicates("native")
            .head(PARENTS_PER_GENERATION)
        )

    def run(self, generations=GENERATIONS):
        parents = self.seed_parents()

        for generation in range(generations):
            proposed = self.propose(parents, generation)
            if proposed.empty:
                print(f"Generation {generation + 1}: no new collision candidates")
                break

            batch = self.select_diverse_batch(proposed)
            if batch.empty:
                print(f"Generation {generation + 1}: diversity constraints left no candidates")
                break

            self.seen.update(zip(batch.native, batch.roman))
            tested = evaluate_candidates(batch)
            self.history = pd.concat([self.history, tested], ignore_index=True)
            self.learn(tested)

            margin_n = int(tested.margin_positive.sum())
            auto_n = int(tested.auto_strict_positive.sum())
            print(
                f"Generation {generation + 1}: tested {len(tested)}, "
                f"margin positives {margin_n}, auto-strict positives {auto_n}"
            )

            display(
                tested.sort_values("reward", ascending=False)[[
                    "native", "roman", "gloss", "collision_token", "predicted_attractor",
                    "collision_score", "semantic_overlap", "roman_margin", "native_margin",
                    "roman_generation_attractor_hits", "native_generation_attractor_hits",
                    "auto_strict_positive",
                ]].head(10).round(3)
            )

            parents = self.choose_parents()

        return self.history.sort_values(
            ["auto_strict_positive", "reward"],
            ascending=[False, False],
        ).reset_index(drop=True)


In [11]:
miner = SISHMiner(lexicon)
results = miner.run(GENERATIONS)

margin_positives = results[results["margin_positive"]].copy()
auto_positives = results[results["auto_strict_positive"]].copy()

print(f"\nMargin-positive candidates: {len(margin_positives)} / {len(results)} tested")
print(
    f"Auto-strict candidates after free-generation check: "
    f"{len(auto_positives)} / {len(results)} tested"
)

if len(auto_positives):
    print(
        "\nIMPORTANT: these are machine-screened candidates, not final verified SISH labels. "
        "Inspect roman_generation_outputs and native_generation_outputs manually."
    )
    display(
        auto_positives[[
            "native", "base_roman", "roman", "gloss",
            "collision_token", "predicted_attractor", "token_pieces",
            "collision_score", "semantic_overlap",
            "roman_margin", "native_margin", "script_contrast",
            "roman_generation_outputs", "native_generation_outputs", "rules",
        ]].head(40)
    )
elif len(margin_positives):
    print(
        "No candidate survived the free-generation check, but the margin-positive rows "
        "are useful hard cases for manual inspection."
    )
    display(
        margin_positives[[
            "native", "roman", "gloss", "collision_token", "predicted_attractor",
            "roman_margin", "native_margin",
            "roman_generation_outputs", "native_generation_outputs",
        ]].head(30)
    )
else:
    print("No margin positives in this run. The tested rows remain useful hard negatives.")


  0%|          | 0/43 [00:00<?, ?it/s]

Generation 1: tested 32, margin positives 3, auto-strict positives 1


,native,roman,gloss,collision_token,predicted_attractor,collision_score,semantic_overlap,roman_margin,native_margin,roman_generation_attractor_hits,native_generation_attractor_hits,auto_strict_positive
13,ಬೆಂಗಾವಲು,beengaavalu,a protecting of a person (as by an escort),been,be,6.270,0.0,4.716,-5.761,0,0,False
14,ಬೆನ್ನುಹುರಿ,beennuhuri,the thick cord of nerve tissue of the central ...,been,be,6.270,0.0,3.060,-5.443,1,0,True
4,ನೋಟು,notu,a brief record of something written down to as...,not,non,6.088,0.0,-7.306,-13.339,0,0,False
17,ಚೌತಿ,chouti,the fourth day of a lunar fortnight,out,failure,5.778,0.0,-12.442,-17.567,0,0,False
5,ನೋಟೀಸು,notisu,"a written letter, note, placard or the like co...",not,non,6.088,0.0,-5.996,-10.752,0,0,False
21,ವಯಸ್ಕ,wayaska,a man of such and such an age,way,style,6.010,0.0,-10.070,-14.767,0,0,False
1,ವಸೂಲು,wasulu,a demanding and getting back the amount lent o...,was,washington,6.820,0.0,-5.463,-10.039,0,0,False
23,ಮಸೀದಿ,madeedi,a Muslim placethat usu,made,do,5.619,0.0,-5.307,-9.149,0,0,False
31,ಮೌನ,mowna,the state or fact of keeping silent,own,have,5.439,0.0,-11.910,-15.119,0,0,False
9,ನಮೂನೆ,namoone,"the shape, outline or configuration of anything",one,i,5.868,0.0,-6.911,-9.143,0,0,False


  0%|          | 0/16 [00:00<?, ?it/s]

Generation 2: tested 12, margin positives 2, auto-strict positives 1


,native,roman,gloss,collision_token,predicted_attractor,collision_score,semantic_overlap,roman_margin,native_margin,roman_generation_attractor_hits,native_generation_attractor_hits,auto_strict_positive
2,ಬೆಂಗಾವಲು,beengavalu,a protecting of a person (as by an escort),been,be,6.270,0.0,4.324,-5.761,1,0,True
3,ನೋಟು,notuu,a brief record of something written down to as...,not,non,6.088,0.0,-3.735,-13.339,0,0,False
5,ಬೆನ್ನುಹುರಿ,beennuhuuri,the thick cord of nerve tissue of the central ...,been,be,6.270,0.0,3.263,-5.443,0,0,False
7,ಚೌತಿ,choutii,the fourth day of a lunar fortnight,out,failure,5.778,0.0,-11.934,-17.567,0,0,False
1,ನೋಟೀಸು,notiisu,"a written letter, note, placard or the like co...",not,non,6.088,0.0,-5.897,-10.752,0,0,False
9,ವಯಸ್ಕ,wayaskaa,a man of such and such an age,way,style,6.010,0.0,-9.703,-14.767,0,0,False
10,ಮಸೀದಿ,madeedii,a Muslim placethat usu,made,do,5.619,0.0,-5.169,-9.149,0,0,False
6,ಸುಗಮಸಂಗೀತ,saangamasangeta,a system of music which is much lighter than t...,get,return,5.979,0.0,-9.192,-13.117,0,0,False
0,ವಸೂಲು,wasuluu,a demanding and getting back the amount lent o...,was,washington,6.820,0.0,-6.756,-10.039,0,0,False
11,ಮೌನ,moowna,the state or fact of keeping silent,own,have,5.439,0.0,-12.575,-15.119,0,0,False


  0%|          | 0/15 [00:00<?, ?it/s]

Generation 3: tested 12, margin positives 1, auto-strict positives 1


,native,roman,gloss,collision_token,predicted_attractor,collision_score,semantic_overlap,roman_margin,native_margin,roman_generation_attractor_hits,native_generation_attractor_hits,auto_strict_positive
10,ಸಾಯಂ,sayaam,in the evening,say,chance,5.589,0.000,-4.722,-11.374,0,0,False
7,ಚಾಣಾಕ್ಷ,canakshha,a man having or showing a clever or shrewd min...,can,tin,6.159,0.000,-12.758,-15.862,0,0,False
2,ಅದ್ಭುತ,adbuthaa,causing wonder,but,just,6.329,0.000,-13.263,-16.045,0,0,False
1,ಬುಟ್ಟಿ,buti,"a container made of interwoven cane, strips of...",but,just,6.329,0.000,-11.546,-14.137,0,0,False
4,ಪದವೀಧರೆ,padhaveedhaare,a woman who has completed a course of study in...,have,rich person,6.409,0.143,-5.768,-8.489,0,0,False
8,ಮಾನ್ಯತೆ,manyatte,esteem for or a sense of the worth or excellen...,many,quantifier can used count nouns often,5.609,0.000,-2.803,-4.818,0,0,False
0,ವಸ್ತುಸಂಗ್ರಹಾಲಯ,wastusangrrahalaya,"an institution, building or room for preservin...",was,washington,6.820,0.000,-8.660,-10.754,0,0,False
3,ನೀವು,newuu,to give a massage to,new,not long duration just relatively recently,6.250,0.062,-8.842,-10.900,0,0,False
11,ಮುಚ್ಚಿ,muchchii,an attendant or messenger,much,large indefinite amount,5.523,0.000,0.679,-1.092,1,0,True
6,ನೆಗೆತ,negetaa,an act of jumping or flying over,get,return,5.979,0.000,-9.862,-10.821,0,0,False



Margin-positive candidates: 6 / 56 tested
Auto-strict candidates after free-generation check: 3 / 56 tested

IMPORTANT: these are machine-screened candidates, not final verified SISH labels. Inspect roman_generation_outputs and native_generation_outputs manually.


,native,base_roman,roman,gloss,collision_token,predicted_attractor,token_pieces,collision_score,semantic_overlap,roman_margin,native_margin,script_contrast,roman_generation_outputs,native_generation_outputs,rules
0,ಬೆಂಗಾವಲು,bengaavalu,beengavalu,a protecting of a person (as by an escort),been,be,▁been | g | av | alu,6.270000,0.0,4.324393,-5.761025,10.085418,"[""miscellaneous items (informal)"", ""valuable (...","[""Bengaluru (in English) - a city in India, fo...",base|vowel_length|vowel_length
1,ಬೆನ್ನುಹುರಿ,bennuhuri,beennuhuri,the thick cord of nerve tissue of the central ...,been,be,▁been | nu | h | uri,6.270000,0.0,3.060138,-5.442658,8.502796,"[""renovation (noun) - the process of renewing ...","[""Bennehuri (in Kannada) means \""a small, roun...",base|vowel_length
2,ಮುಚ್ಚಿ,muchchi,muchchii,an attendant or messenger,much,large indefinite amount,▁much | ch | ii,5.522879,0.0,0.679084,-1.092313,1.771397,"[""\""Muchchii\"" does not appear to be a standar...","[""Discard, throw away, or abandon."", ""Discard,...",base|vowel_length


In [12]:
output_dir = Path("/content/sish_run")
output_dir.mkdir(exist_ok=True)

results.to_csv(output_dir / "all_candidates.csv", index=False)
margin_positives.to_csv(output_dir / "margin_positive_candidates.csv", index=False)
auto_positives.to_csv(output_dir / "manual_review_candidates.csv", index=False)
screen_details.to_csv(output_dir / "model_screen.csv", index=False)
lexicon.to_csv(output_dir / "source_lexicon.csv", index=False)

metadata = {
    "model_id": MODEL_ID,
    "seed": SEED,
    "lexicon_entries": len(lexicon),
    "lexicon_provenance": {
        "meanings": "Alar Kannada-English dictionary by V. Krishna (ODC-ODbL)",
        "romanizations": sorted(ROMANIZATION_SOURCES),
        "aksharantar_revision": AKSHARANTAR_REVISION,
        "alar_revision": ALAR_REVISION,
        "indowordnet_revision": INDOWORDNET_REVISION,
        "requires_indowordnet": REQUIRE_INDOWORDNET,
        "requires_indiccorp_attestation": REQUIRE_INDICCORP_ATTESTATION,
        "model_generated_native_words": False,
        "model_generated_gold_glosses": False,
    },
    "screen": {
        "words": SCREEN_WORDS,
        "choices": SCREEN_CHOICES,
        "min_native_accuracy_warning_threshold": MIN_NATIVE_SCREEN_ACCURACY,
        "min_script_gap_warning_threshold": MIN_SCRIPT_GAP,
    },
    "generations": GENERATIONS,
    "prompt_templates": PROMPT_TEMPLATES,
    "labeled_control_template": LABELED_CONTROL_TEMPLATE,
    "collision_scoring": {
        "minimum_english_zipf": MIN_ENGLISH_ZIPF,
        "kannada_reference": "token-piece counts from attested Aksharantar Romanizations",
        "requires_wordnet_sense": True,
    },
    "semantic_filter": {
        "max_semantic_overlap": MAX_SEMANTIC_OVERLAP,
        "shared_concept_stems_rejection_threshold": MIN_SHARED_CONCEPT_STEMS_TO_REJECT,
        "literal_collision_token_used_as_likelihood_attractor": False,
        "attractor_source": "WordNet synonym/hypernym/definition paraphrase",
    },
    "margin_positive_criterion": {
        "roman_prompt_wins_required": PROMPT_WINS_REQUIRED,
        "native_prompt_wins_required": PROMPT_WINS_REQUIRED,
        "roman_mean_margin_positive": True,
        "native_mean_margin_negative": True,
    },
    "auto_strict_criterion": {
        "requires_margin_positive": True,
        "roman_free_generation_attractor_hits_required": GENERATION_WINS_REQUIRED,
        "native_free_generation_attractor_hits_required": 0,
        "manual_review_still_required": True,
    },
    "diversity": {
        "max_per_attractor_per_generation": MAX_PER_ATTRACTOR_PER_GENERATION,
        "max_tests_per_native": MAX_TESTS_PER_NATIVE,
        "novelty_bonus": NOVELTY_BONUS,
    },
}

(output_dir / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

archive = Path("/content/sish_run_v2.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in output_dir.iterdir():
        zf.write(path, path.name)

print(f"Saved {archive}. Download it from Colab's Files panel.")
print("manual_review_candidates.csv contains the strongest machine-screened cases.")


Saved /content/sish_run_v2.zip. Download it from Colab's Files panel.
manual_review_candidates.csv contains the strongest machine-screened cases.
